# 03 — Merge model output (one experiment at a time)

ECHAM6.3 writes one file per model year (1976–1999) and per output stream.
This notebook merges the 24 yearly files of **one** experiment into one file
per stream, with the file name every later notebook expects:

`{YEAR}_{RUN}_echam6_{STREAM}_mm_1976-1999.nc` in the El Niño or La Niña output folder.

Streams: **ATM** (3-D pressure-level fields, e.g. `omega`, `st`) and **BOT**
(2-D surface and top-of-atmosphere fluxes). Set the four values in Step 1
and run; repeat for each of the 69 experiments (23 events × 3 runs).

In [1]:
import sys
sys.path.insert(0, "../src")
import config as cfg

import os
import xarray as xr

## Step 1 — which experiment

In [2]:
YEAR = 2012                 # event year (01's table)
EVENT = "La Niña"           # "El Niño" or "La Niña"
RUN = "PATTERN"             # "FULL", "UNIFORM" or "PATTERN"
EXPERIMENT_ID = "mca0069"   # ECHAM experiment id of this run

RAW_OUTPUT_DIR = os.path.join(cfg.ECHAM_EXPERIMENTS_DIR, EXPERIMENT_ID, "outdata", "echam6")
OUT_DIR = cfg.ECHAM_OUTPUT_ELNINO_DIR if EVENT == "El Niño" else cfg.ECHAM_OUTPUT_LANINA_DIR
os.makedirs(OUT_DIR, exist_ok=True)

## Step 2 — merge the yearly files, one output file per stream

In [3]:
for stream in ["ATM", "BOT"]:
    yearly_files = []
    for y in range(1976, 2000):
        f = os.path.join(RAW_OUTPUT_DIR, f"{EXPERIMENT_ID}_echam6_{stream}_mm_{y}.nc")
        if os.path.exists(f):
            yearly_files.append(xr.open_dataset(f))
    if not yearly_files:
        print(f"no files found for {stream} in {RAW_OUTPUT_DIR} -- check EXPERIMENT_ID")
        continue
    if len(yearly_files) != 24:
        print(f"WARNING: {stream} has {len(yearly_files)} of 24 yearly files")
    merged = xr.concat(yearly_files, dim="time", join="override")
    out_path = os.path.join(OUT_DIR, f"{YEAR}_{RUN}_echam6_{stream}_mm_1976-1999.nc")
    merged.to_netcdf(out_path)
    print(f"saved {out_path} ({len(yearly_files)} yearly files)")

saved /work/mh1421/m301201/data/echam-output/lanina/2012_PATTERN_echam6_ATM_mm_1976-1999.nc (24 yearly files)
saved /work/mh1421/m301201/data/echam-output/lanina/2012_PATTERN_echam6_BOT_mm_1976-1999.nc (24 yearly files)
